# HydrAI — Bayesian Optimisation: Surrogate vs Cantera Validation

Use Optuna GP (Gaussian Process sampler) to optimise reactor inlet conditions for maximum
olefins yield, running separately with two fast surrogates:

1. **MLP** — `SimpleNN` exit-plane model from Main_6
2. **SR** — symbolic expressions distilled from the MLP by Main_9

Both optima are then validated with a real Cantera PFR simulation so the surrogate
optimisation error can be quantified.

**Prerequisites:** run Main_6 (`IF_MODEL_EXPORT = True`) and Main_9 (`IF_SR_EXPORT = True`) first.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 1. SETUP
# ════════════════════════════════════════════════════════════════════════════
import sys
import os
from pathlib import Path
import importlib.util
import json
import warnings
warnings.filterwarnings('ignore')

# Resolve project root (works whether cwd is repo root or notebooks/)
current_dir = Path(os.getcwd())
if (current_dir / 'src').exists():
    project_root = current_dir
elif (current_dir.parent / 'src').exists():
    project_root = current_dir.parent
else:
    project_root = current_dir
os.chdir(project_root)

# IMPORTANT: import cantera BEFORE adding src/ to sys.path to avoid shadowing
import cantera as ct
print(f'Cantera {ct.__version__}')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import torch
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

from src.models import SimpleNN
from src.utils.plot_style import setup_matplotlib
from src.ml.data_generation import TrainingDataGenerator

print(f'Optuna {optuna.__version__}')
print(f'PyTorch {torch.__version__}')
print(f'Project root: {project_root}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 2. CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════

# --- Optimisation target ---
# Column name in the MLP target_cols to maximise at reactor exit.
OPT_TARGET   = 'Y_lump_olefins'

# --- Optuna GP budget ---
N_TRIALS     = 150        # per surrogate study
RANDOM_STATE = 42

# --- Inlet condition search bounds (training-data domain) ---
BOUNDS = {
    'T_K':    (800.0,    900.0),     # inlet temperature [K]
    'P_Pa':   (1.5e5,    3.5e5),     # inlet pressure [Pa]
    'L_m':    (10.0,     15.0),      # reactor length [m]
    'D_m':    (0.025,    0.040),     # reactor diameter [m]
    'mdot':   (0.05,     0.10),      # mass flow rate [kg/s]
    'q_Wm2':  (1.0e5,    2.5e5),    # wall heat flux [W/m²]
}

# --- Cantera reactant ---
REACTANT_KEY = 'n-hexane'

# --- Artefact stems ---
MLP_STEM = 'simple_nn_exit'         # from Main_6
SR_DIR   = project_root / 'models' / 'sr_exit'

# --- Flags ---
IF_PLOT_SHOWN  = True
IF_PLOT_EXPORT = True
FIGS_DIR = project_root / 'outputs' / 'figures' / 'Main_10_optimisation'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Target:     {OPT_TARGET}')
print(f'N_TRIALS:   {N_TRIALS}')
print(f'Reactant:   {REACTANT_KEY}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 3. LOAD MLP ARTEFACTS (Main_6 SimpleNN)
# ════════════════════════════════════════════════════════════════════════════
models_dir = project_root / 'models'

manifest_path = models_dir / f'{MLP_STEM}_manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError(
        f'{manifest_path} not found — run Main_6 with IF_MODEL_EXPORT=True first.'
    )

with open(manifest_path) as f:
    mlp_manifest = json.load(f)

arch        = mlp_manifest['architecture']
inlet_cols  = mlp_manifest.get('inlet_cols') or mlp_manifest.get('feature_cols', [])
target_cols = mlp_manifest['target_cols']

mlp_scalers = joblib.load(models_dir / f'{MLP_STEM}_scalers.joblib')
mlp_scaler_X = mlp_scalers['scaler_X']
mlp_scaler_y = mlp_scalers.get('scaler_y')

mlp_model = SimpleNN(
    arch['in_features'], arch['h1'], arch['h2'], arch['h3'],
    arch['out_features'], dropout=arch['dropout'],
)
mlp_model.load_state_dict(torch.load(
    models_dir / f'{MLP_STEM}_state_dict.pt',
    map_location='cpu', weights_only=True,
))
mlp_model.eval()

if OPT_TARGET not in target_cols:
    raise ValueError(f'OPT_TARGET "{OPT_TARGET}" not in target_cols: {target_cols}')
opt_target_idx = target_cols.index(OPT_TARGET)

print(f'MLP loaded  — inputs: {len(inlet_cols)}, outputs: {len(target_cols)}')
print(f'Target col [{opt_target_idx}]: {OPT_TARGET}')


def predict_mlp(T_K, P_Pa, L_m, D_m, mdot, q_Wm2) -> float:
    """Return MLP prediction for OPT_TARGET given inlet conditions."""
    row = {
        'initial_temperature_K': T_K,
        'initial_pressure_Pa':   P_Pa,
        'reactor_length_m':      L_m,
        'reactor_diameter_m':    D_m,
        'mass_flow_rate_kgps':   mdot,
        'heat_flux_Wm2':         q_Wm2,
    }
    X = np.array([[row.get(c, 0.0) for c in inlet_cols]], dtype=float)
    X_s = mlp_scaler_X.transform(X)
    with torch.no_grad():
        y_s = mlp_model(torch.tensor(X_s, dtype=torch.float32)).numpy()
    y = mlp_scaler_y.inverse_transform(y_s) if mlp_scaler_y is not None else y_s
    return float(y[0, opt_target_idx])

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 4. LOAD SR ARTEFACTS (Main_9 symbolic equations)
# ════════════════════════════════════════════════════════════════════════════
sr_eq_path = SR_DIR / 'sr_exit_equations.py'
if not sr_eq_path.exists():
    raise FileNotFoundError(
        f'{sr_eq_path} not found — run Main_9 with IF_SR_EXPORT=True first.'
    )

spec = importlib.util.spec_from_file_location('sr_exit_equations', sr_eq_path)
sr_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sr_module)

# Build the target function name from OPT_TARGET (mirrors Main_9 export logic)
sr_fn_name = 'predict_' + OPT_TARGET.replace('.', '_').replace(' ', '_')
if not hasattr(sr_module, sr_fn_name):
    available = [x for x in dir(sr_module) if x.startswith('predict_')]
    raise AttributeError(
        f'{sr_fn_name} not found in SR module. Available: {available}'
    )
sr_fn = getattr(sr_module, sr_fn_name)

# Load SR manifest for metadata
sr_manifest_path = SR_DIR / 'sr_exit_manifest.json'
sr_meta = json.loads(sr_manifest_path.read_text()) if sr_manifest_path.exists() else {}
sr_inlet_cols = sr_meta.get('inlet_cols', inlet_cols)  # fallback to MLP inlet cols

print(f'SR function loaded: {sr_fn_name}')
print(f'SR inlet cols: {sr_inlet_cols}')


def predict_sr(T_K, P_Pa, L_m, D_m, mdot, q_Wm2) -> float:
    """Return SR expression prediction for OPT_TARGET given inlet conditions."""
    arg_map = {
        'initial_temperature_K': T_K,
        'initial_pressure_Pa':   P_Pa,
        'reactor_length_m':      L_m,
        'reactor_diameter_m':    D_m,
        'mass_flow_rate_kgps':   mdot,
        'heat_flux_Wm2':         q_Wm2,
    }
    args = [arg_map.get(c, 0.0) for c in sr_inlet_cols]
    try:
        val = float(sr_fn(*args))
    except Exception:
        val = float('nan')
    return val

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 5. OPTUNA GP STUDY — MLP SURROGATE
# ════════════════════════════════════════════════════════════════════════════

def mlp_objective(trial: optuna.Trial) -> float:
    T_K   = trial.suggest_float('T_K',   *BOUNDS['T_K'])
    P_Pa  = trial.suggest_float('P_Pa',  *BOUNDS['P_Pa'])
    L_m   = trial.suggest_float('L_m',   *BOUNDS['L_m'])
    D_m   = trial.suggest_float('D_m',   *BOUNDS['D_m'])
    mdot  = trial.suggest_float('mdot',  *BOUNDS['mdot'])
    q_Wm2 = trial.suggest_float('q_Wm2', *BOUNDS['q_Wm2'])
    return predict_mlp(T_K, P_Pa, L_m, D_m, mdot, q_Wm2)


study_mlp = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.GPSampler(seed=RANDOM_STATE),
    study_name='mlp_bo',
)
study_mlp.optimize(mlp_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_mlp  = study_mlp.best_params
best_mlp_val = study_mlp.best_value
print(f'\nMLP best {OPT_TARGET} = {best_mlp_val:.6f}')
for k, v in best_mlp.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 6. OPTUNA GP STUDY — SR SURROGATE
# ════════════════════════════════════════════════════════════════════════════

def sr_objective(trial: optuna.Trial) -> float:
    T_K   = trial.suggest_float('T_K',   *BOUNDS['T_K'])
    P_Pa  = trial.suggest_float('P_Pa',  *BOUNDS['P_Pa'])
    L_m   = trial.suggest_float('L_m',   *BOUNDS['L_m'])
    D_m   = trial.suggest_float('D_m',   *BOUNDS['D_m'])
    mdot  = trial.suggest_float('mdot',  *BOUNDS['mdot'])
    q_Wm2 = trial.suggest_float('q_Wm2', *BOUNDS['q_Wm2'])
    val   = predict_sr(T_K, P_Pa, L_m, D_m, mdot, q_Wm2)
    return float(val) if np.isfinite(val) else -1e9


study_sr = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.GPSampler(seed=RANDOM_STATE),
    study_name='sr_bo',
)
study_sr.optimize(sr_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_sr  = study_sr.best_params
best_sr_val = study_sr.best_value
print(f'\nSR best {OPT_TARGET} = {best_sr_val:.6f}')
for k, v in best_sr.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 7. OPTIMISATION HISTORY PLOT
# ════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=False)
setup_matplotlib(axes)

for ax, study, label, color in [
    (axes[0], study_mlp, 'MLP surrogate', 'b'),
    (axes[1], study_sr,  'SR surrogate',  'r'),
]:
    trials = study.trials
    vals   = [t.value for t in trials]
    best_so_far = np.maximum.accumulate(vals)

    ax.scatter(range(len(vals)), vals, s=6, alpha=0.5, c=color, label='trial value')
    ax.plot(range(len(best_so_far)), best_so_far, color=color, lw=1.5, label='best so far')
    ax.set_xlabel('Trial')
    ax.set_ylabel(OPT_TARGET)
    ax.set_title(f'GP-BO history — {label}')
    ax.legend(fontsize=8)

plt.tight_layout()
if IF_PLOT_EXPORT:
    fig.savefig(FIGS_DIR / 'bo_history.png', dpi=200, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 8. BEST PARAMETERS TABLE
# ════════════════════════════════════════════════════════════════════════════
param_labels = {
    'T_K':   'T_in [K]',
    'P_Pa':  'P_in [bar]',
    'L_m':   'L [m]',
    'D_m':   'D [m]',
    'mdot':  'ṁ [kg/s]',
    'q_Wm2': 'q [kW/m²]',
}
scale = {
    'T_K':   1.0,
    'P_Pa':  1e-5,     # → bar
    'L_m':   1.0,
    'D_m':   1.0,
    'mdot':  1.0,
    'q_Wm2': 1e-3,     # → kW/m²
}

rows = []
for k, label in param_labels.items():
    rows.append({
        'Parameter': label,
        'MLP optimal': f'{best_mlp[k] * scale[k]:.4g}',
        'SR optimal':  f'{best_sr[k]  * scale[k]:.4g}',
    })
rows.append({
    'Parameter': f'{OPT_TARGET} (surrogate)',
    'MLP optimal': f'{best_mlp_val:.6f}',
    'SR optimal':  f'{best_sr_val:.6f}',
})
df_best = pd.DataFrame(rows)
print(df_best.to_string(index=False))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 9. CANTERA VALIDATION
# Run Cantera at both surrogate optima and extract exit-plane values.
# ════════════════════════════════════════════════════════════════════════════
import logging
logging.getLogger('cantera').setLevel(logging.ERROR)

# Species that map to Y_lump_olefins (primary olefin products in steam cracking)
OLEFIN_SPECIES = ['Y_C2H4', 'Y_C3H6', 'Y_C4H6', 'Y_C4H8']


def _run_cantera_and_extract(params: dict, sim_id: int) -> dict:
    """Run a single Cantera PFR and return exit-plane key values."""
    gen = TrainingDataGenerator(output_dir=str(project_root / 'data' / 'training'), disable_plots=True)
    df = gen.run_single_simulation(REACTANT_KEY, params, sim_id=sim_id)
    if df is None or len(df) == 0:
        return {'error': 'Cantera simulation failed'}
    row = df.iloc[-1]   # exit plane
    result = {'T_exit_K': float(row.get('temperature_K', row.get('T', float('nan'))))}
    # Sum olefin mass fractions
    olefin_total = sum(float(row.get(sp, 0.0)) for sp in OLEFIN_SPECIES)
    result['Y_lump_olefins_cantera'] = olefin_total
    # Also capture any direct Y_lump_ columns if present
    for col in df.columns:
        if col.startswith('Y_lump_'):
            result[col + '_cantera'] = float(row[col])
    return result


# Build params dicts from Optuna best
def _optuna_to_params(best: dict) -> dict:
    return {
        'temperature_K':      best['T_K'],
        'pressure_Pa':        best['P_Pa'],
        'length_m':           best['L_m'],
        'diameter_m':         best['D_m'],
        'mass_flow_rate_kgps': best['mdot'],
        'heat_flux_Wm2':      best['q_Wm2'],
    }


print('Running Cantera at MLP-optimal conditions ...')
ct_mlp = _run_cantera_and_extract(_optuna_to_params(best_mlp), sim_id=1)
print(f'  Done. Y_lump_olefins (Cantera) = {ct_mlp.get("Y_lump_olefins_cantera", ct_mlp.get("Y_lump_olefins", "N/A")):.6f}')

print('Running Cantera at SR-optimal conditions ...')
ct_sr = _run_cantera_and_extract(_optuna_to_params(best_sr), sim_id=2)
print(f'  Done. Y_lump_olefins (Cantera) = {ct_sr.get("Y_lump_olefins_cantera", ct_sr.get("Y_lump_olefins", "N/A")):.6f}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 10. COMPARISON PLOT — surrogate prediction vs Cantera ground truth
# ════════════════════════════════════════════════════════════════════════════

def _get_cantera_olefins(ct_result: dict) -> float:
    if 'Y_lump_olefins_cantera' in ct_result:
        return ct_result['Y_lump_olefins_cantera']
    return ct_result.get('Y_lump_olefins', float('nan'))


val_mlp_surrogate = best_mlp_val
val_sr_surrogate  = best_sr_val
val_mlp_cantera   = _get_cantera_olefins(ct_mlp)
val_sr_cantera    = _get_cantera_olefins(ct_sr)

err_mlp = abs(val_mlp_surrogate - val_mlp_cantera) / (abs(val_mlp_cantera) + 1e-12) * 100
err_sr  = abs(val_sr_surrogate  - val_sr_cantera)  / (abs(val_sr_cantera)  + 1e-12) * 100

# ── Bar chart ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4.5))
setup_matplotlib([ax])

labels  = ['MLP (surrogate)', 'MLP (Cantera)', 'SR (surrogate)', 'SR (Cantera)']
values  = [val_mlp_surrogate, val_mlp_cantera, val_sr_surrogate, val_sr_cantera]
colors  = ['b', 'b', 'r', 'r']
hatches = ['', '///', '', '///']

bars = ax.bar(labels, values, color='white', edgecolor=colors, hatch=hatches, linewidth=1.4)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)

ax.set_ylabel(OPT_TARGET)
ax.set_title(f'Surrogate optimum vs Cantera ground truth — {OPT_TARGET}')
ax.tick_params(axis='x', labelrotation=15)

ax.annotate(f'MLP error: {err_mlp:.1f}%', xy=(0.5, 0.02), xycoords='axes fraction',
            color='b', fontsize=8, ha='center')
ax.annotate(f'SR error: {err_sr:.1f}%',   xy=(0.5, 0.08), xycoords='axes fraction',
            color='r', fontsize=8, ha='center')

plt.tight_layout()
if IF_PLOT_EXPORT:
    fig.savefig(FIGS_DIR / 'surrogate_vs_cantera.png', dpi=200, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

print(f'MLP → surrogate: {val_mlp_surrogate:.6f}  |  Cantera: {val_mlp_cantera:.6f}  |  error: {err_mlp:.2f}%')
print(f'SR  → surrogate: {val_sr_surrogate:.6f}  |  Cantera: {val_sr_cantera:.6f}  |  error: {err_sr:.2f}%')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 11. PARAMETER COMPARISON PLOT
# Show how the two surrogate optima differ in parameter space.
# ════════════════════════════════════════════════════════════════════════════
param_keys   = list(param_labels.keys())
param_names  = list(param_labels.values())
mlp_norm = [(best_mlp[k] - BOUNDS[k][0]) / (BOUNDS[k][1] - BOUNDS[k][0]) for k in param_keys]
sr_norm  = [(best_sr[k]  - BOUNDS[k][0]) / (BOUNDS[k][1] - BOUNDS[k][0]) for k in param_keys]

x = np.arange(len(param_keys))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
setup_matplotlib([ax])

ax.bar(x - w/2, mlp_norm, width=w, facecolor='white', edgecolor='b', label='MLP optimal', linewidth=1.2)
ax.bar(x + w/2, sr_norm,  width=w, facecolor='white', edgecolor='r', hatch='///', label='SR optimal', linewidth=1.2)
ax.axhline(0.0, color='k', lw=0.8)
ax.axhline(1.0, color='k', lw=0.8, ls='--', label='domain boundary')
ax.set_xticks(x)
ax.set_xticklabels(param_names, fontsize=9)
ax.set_ylabel('Normalised value (0 = min, 1 = max)')
ax.set_title('MLP vs SR optimal parameters (normalised to training domain)')
ax.legend(fontsize=8)

plt.tight_layout()
if IF_PLOT_EXPORT:
    fig.savefig(FIGS_DIR / 'optimal_params_comparison.png', dpi=200, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

---

## Summary

| | MLP (Main_6) | SR (Main_9) |
|---|---|---|
| Surrogate type | PyTorch `SimpleNN` | PySR closed-form expression |
| Optimiser | Optuna `GPSampler` | Optuna `GPSampler` |
| Trials | `N_TRIALS` | `N_TRIALS` |
| Objective | max `Y_lump_olefins` at exit | max `Y_lump_olefins` at exit |

### Interpretation
- **Surrogate error**: difference between the value the surrogate predicted at its optimum and what Cantera computes at the same conditions. Lower = the surrogate is reliable in this region.
- **Cross-validation**: running both optima through Cantera shows which surrogate found conditions that are truly better — not just ones that exploit surrogate extrapolation artefacts.
- **SR advantage**: the SR expression is differentiable in closed form, making it suitable for gradient-based optimisation and embedding in process simulators without a PyTorch runtime.